# SecondLook | 10 AI Analyst Agent

**Goal:** a chat page where a manager asks questions in plain English ("Which persona has the most repayers we decline?") and Claude answers by **writing SQL and calling our own tools**, showing its work.

```
question ─► Claude Haiku 5.5 ──► tool call ──► our Python runs it ──► result ──► Claude ─► ... ─► answer
                 │  run_sql(query)            read-only SQLite (portfolio, personas, fairness)
                 │  explain_applicant(id)     SHAP reasons (notebook 04)
                 │  path_to_approval(id)      re-scored loan terms (notebook 06)
                 └  fairness_summary(group)   audit with 95% intervals (notebook 08)
```

**Agentic AI:** the model decides *which* tool to call and *what* SQL to write, reads the result, and decides whether it needs another step. We wrote the loop ourselves (`secondlook.agent.ask`, about 40 lines), so every step is visible: it stops at the final answer or after **6 tool calls**.

**Safety:** `run_sql` opens the file read-only (`mode=ro`), accepts one SELECT only, caps results at 200 rows and returns errors as text so Claude can fix its own query (tests in `tests/test_agent.py`). The app adds a 10-question limit per session to protect the API key.

## 1. Setup

In [1]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "secondlook").exists())
sys.path.insert(0, str(ROOT / "src"))

import os
from dotenv import load_dotenv
import anthropic

from secondlook import agent as AG
from secondlook import path as P
from secondlook.letters import MODEL

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

load_dotenv(ROOT / ".env")
BUNDLE = ROOT / "app" / "demo_data"
toolbox = AG.Toolbox.from_bundle(BUNDLE)
client = anthropic.Anthropic()
print(f"Model: {MODEL} · max tool calls: {AG.MAX_TOOL_CALLS} · max rows: {AG.MAX_ROWS}")

Model: claude-haiku-5-5 · max tool calls: 6 · max rows: 200


## 2. Safety checks: refused safely

In [2]:
for q in ["DROP TABLE portfolio", "UPDATE portfolio SET pd = 0", "SELECT 1; DROP TABLE portfolio",
          "SELECT * FROM portfolio LIMIT 10000"]:
    out = AG.run_sql(q, toolbox.db_path)
    print(f"{q[:40]:42} -> {out[:90] if out.startswith('Error') else 'returned ' + str(json.loads(out)['row_count']) + ' rows (capped)'}")

DROP TABLE portfolio                       -> Error: only SELECT queries are allowed.
UPDATE portfolio SET pd = 0                -> Error: only SELECT queries are allowed.
SELECT 1; DROP TABLE portfolio             -> Error: only one statement is allowed per call.
SELECT * FROM portfolio LIMIT 10000        -> returned 200 rows (capped)


## 3. Ten test questions with answers computed by hand

The expected answers are computed with **pandas from the bundle files** (`portfolio.parquet`, `personas.parquet`), independently of the SQLite database the agent queries. A question counts as correct when the answer states the expected number (within rounding) or names the expected persona.

In [3]:
port = pd.read_parquet(BUNDLE / "portfolio.parquet")
pers = pd.read_parquet(BUNDLE / "personas.parquet")
cut = toolbox.meta["cutoff"]
port["decision3"] = np.select([port["PD_LGBM"] < cut, port["PD_LGBM"] < cut + 0.05], ["approve", "under_review"], "decline")
thin = port["HAS_BUREAU_HISTORY"] == 0
na = pers.copy()

# An under-review applicant from the app bundle with a path to approval, for the path tool
apps = toolbox.applicants
bundle_ids = set(apps["SK_ID_CURR"])
review_ids = port[(port["decision3"] == "under_review") & port["SK_ID_CURR"].isin(bundle_ids)]["SK_ID_CURR"]
path_id, path_rec = None, None
for sid in review_ids:
    p = P.path_to_approval(apps[apps["SK_ID_CURR"] == sid], toolbox.booster, toolbox.meta, cut)
    if p["status"] == "path" and p["borrow_less"] is not None:
        path_id, path_rec = int(sid), p["borrow_less"]
        break

repaid_by_persona = na[na["TARGET"] == 0].groupby("persona").size()
QUESTIONS = [
    ("What is the approval rate of thin-file applicants?", "pct", (port.loc[thin, "decision3"] == "approve").mean()),
    ("How many applicants are under review?", "count", (port["decision3"] == "under_review").sum()),
    ("What share of under-review applicants actually repaid their loan?", "pct",
     1 - port.loc[port["decision3"] == "under_review", "TARGET"].mean()),
    ("Which persona has the most repayers among the applicants we do not approve?", "text", repaid_by_persona.idxmax()),
    ("What is the default rate among approved applicants?", "pct", port.loc[port["decision3"] == "approve", "TARGET"].mean()),
    ("What is the average loan amount of declined applicants?", "amount", port.loc[port["decision3"] == "decline", "AMT_CREDIT"].mean()),
    ("Among applicants who repaid, what is the approval rate for women?", "pct",
     (port.loc[(port["TARGET"] == 0) & (port["CODE_GENDER"] == "F"), "decision3"] == "approve").mean()),
    ("Of the not-approved applicants who have a path to approval, what share actually repaid?", "pct",
     1 - na.loc[na["has_path"] == True, "TARGET"].mean()),
    ("How many thin-file applicants were declined outright but actually repaid?", "count",
     ((port["decision3"] == "decline") & thin & (port["TARGET"] == 0)).sum()),
    (f"Applicant {path_id} is under review. If they keep the same number of payments, what is the largest loan amount "
     "that would get them approved?", "amount", path_rec["loan_amount"]),
]
pd.DataFrame(QUESTIONS, columns=["question", "kind", "expected"])

,question,kind,expected
0,What is the approval rate of thin-file applicants?,pct,0.827028
1,How many applicants are under review?,count,2611
2,What share of under-review applicants actually repaid their loan?,pct,0.797396
3,Which persona has the most repayers among the applicants we do not approve?,text,"Newer worker, ordinary loan"
4,What is the default rate among approved applicants?,pct,0.053153
5,What is the average loan amount of declined applicants?,amount,508679.96875
6,"Among applicants who repaid, what is the approval rate for women?",pct,0.9222
7,"Of the not-approved applicants who have a path to approval, what share actually repaid?",pct,0.770483
8,How many thin-file applicants were declined outright but actually repaid?,count,644
9,"Applicant 173037 is under review. If they keep the same number of payments, what is the largest loan amount that wou...",amount,432990.0


In [4]:
import re

def numbers(text):
    return [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", text)]

def correct(answer, kind, expected):
    nums = numbers(answer)
    if kind == "text":
        return str(expected).lower() in answer.lower()
    if kind == "pct":       # stated as a percentage with one decimal, so allow 0.1 points of rounding
        return any(abs(x - expected * 100) <= 0.11 for x in nums)
    if kind == "count":
        return any(x == expected for x in nums)
    return any(abs(x - expected) <= max(1.0, 0.005 * expected) for x in nums)    # amount: within 0.5%

results = []
for q, kind, expected in QUESTIONS:
    r = AG.ask(client, q, toolbox)
    results.append({"question": q, "expected": expected, "answer": r.answer, "correct": correct(r.answer, kind, expected),
                    "tool calls": len(r.steps), "tools": ", ".join(s["tool"] for s in r.steps), "cost_usd": r.cost_usd,
                    "seconds": r.seconds, "sql": r.sql})
res = pd.DataFrame(results)
print(f"Correct: {res['correct'].sum()} of {len(res)} · total cost ${res['cost_usd'].sum():.4f} "
      f"(${res['cost_usd'].mean():.4f} per question) · median {res['seconds'].median():.0f} s per answer")
res[["question", "expected", "correct", "tool calls", "tools", "cost_usd"]]

Correct: 10 of 10 · total cost $0.0060 ($0.0006 per question) · median 2 s per answer


,question,expected,correct,tool calls,tools,cost_usd
0,What is the approval rate of thin-file applicants?,0.827028,True,1,run_sql,0.000567
1,How many applicants are under review?,2611,True,1,run_sql,0.000550
2,What share of under-review applicants actually repaid their loan?,0.797396,True,1,run_sql,0.000579
3,Which persona has the most repayers among the applicants we do not approve?,"Newer worker, ordinary loan",True,1,run_sql,0.000604
4,What is the default rate among approved applicants?,0.053153,True,1,run_sql,0.000537
5,What is the average loan amount of declined applicants?,508679.96875,True,1,run_sql,0.000546
6,"Among applicants who repaid, what is the approval rate for women?",0.9222,True,1,run_sql,0.000587
7,"Of the not-approved applicants who have a path to approval, what share actually repaid?",0.770483,True,1,run_sql,0.000565
8,How many thin-file applicants were declined outright but actually repaid?,644,True,1,run_sql,0.000608
9,"Applicant 173037 is under review. If they keep the same number of payments, what is the largest loan amount that wou...",432990.0,True,2,"path_to_approval, explain_applicant",0.000903


### Read the answers, not just the score

In [5]:
for r in results:
    print(f"Q: {r['question']}\nExpected: {r['expected']:.4g}" if not isinstance(r["expected"], str) else
          f"Q: {r['question']}\nExpected: {r['expected']}")
    print(f"A ({'correct' if r['correct'] else 'WRONG'}): {r['answer']}")
    for s in r["sql"]:
        print("   SQL:", " ".join(s.split())[:160])
    print()

Q: What is the approval rate of thin-file applicants?
Expected: 0.827
A (correct): Thin-file applicants have an approval rate of 82.7% (7,260 of 8,776), compared with 89.1% for applicants who have a bureau history. That leaves thin-file applicants about 6.4 points less likely to be approved.
   SQL: SELECT has_bureau_history, COUNT(*) AS applicants, ROUND(100.0*SUM(CASE WHEN decision='approve' THEN 1 ELSE 0 END)/COUNT(*),1) AS approval_rate_pct FROM portfol

Q: How many applicants are under review?
Expected: 2611
A (correct): There are 2,611 applicants under review, which is 4.2% of the 61,503-applicant portfolio. These are applicants with a predicted default risk between 17% and 22%, so they need manual review rather than an automatic decision.
   SQL: SELECT COUNT(*) AS under_review_count, ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM portfolio),1) AS pct_of_portfolio FROM portfolio WHERE decision='under_review'

Q: What share of under-review applicants actually repaid their loan?
Expec

In [6]:
res.drop(columns=["sql"]).to_json(ROOT / "data" / "processed" / "agent_eval.json", orient="records", indent=1)
print("Saved data/processed/agent_eval.json")

Saved data/processed/agent_eval.json


## Takeaways

1. **10 of 10 test questions answered correctly** (the guide's bar is 9), each checked against a number computed independently with pandas. Nine needed one SQL query; the applicant question used the path-to-approval tool.
2. **Cheap and fast with Claude Haiku 5.5:** about **$0.0006 per question** ($0.006 for all ten) and about 2 seconds per answer, so we did not need Sonnet.
3. **Every unsafe query is refused safely** (DROP, UPDATE, two statements); a 10,000-row query is capped at 200 rows. The database is also opened read-only, so SQLite itself would refuse a write.
4. **Reading the answers found two weaknesses the score hid.** In an earlier run the agent headlined an *interpolated* loan amount it made up between two tool results; we added a rule to report only numbers a tool returned, and it now quotes the tested amount and says the tool searches in 5% steps. In this run, one answer states the right number (79.7% of under-review applicants repaid) but misreads it ("roughly one in five ... would have been a good borrower"; it is four in five). The numbers are reliable, the sentences around them need a human eye, which is why every answer in the app shows its SQL.
5. **Limits:** ten questions is a small test, and the agent only knows what is in the three tables. Questions about individual applicants work only for the 300 applicants in the app bundle.